# Train YOLOv8n Fall Detection tren Google Colab

Notebook nay dung de train model YOLOv8n cho bai toan phat hien te nga.

Input can upload len Colab:
- `yolo_coffee_room.zip`: dataset YOLO da dong goi san.

Output sau train:
- `best.pt`: model YOLO tot nhat, dung cho `src/predict_video.py`.

## 1. Kiem tra GPU

Tren Colab, vao `Runtime` -> `Change runtime type` -> chon `T4 GPU` truoc khi chay cell nay.

In [ ]:
!nvidia-smi

## 2. Cai thu vien can thiet

In [ ]:
!pip install ultralytics opencv-python -q

## 3. Upload dataset YOLO zip

Upload file:

```text
yolo_coffee_room.zip
```

File nay nam tren may local tai:

```text
C:\Users\ASUS-PRO\Documents\lezzi_datasets\data\yolo_coffee_room.zip
```

In [ ]:
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

## 4. Giai nen dataset

In [ ]:
import zipfile
from pathlib import Path

zip_name = list(uploaded.keys())[0]
extract_root = Path('/content')

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_root)

print('Da giai nen:', zip_name)

## 5. Kiem tra cau truc dataset

Sau khi giai nen, dataset nen nam o:

```text
/content/yolo_coffee_room
```

In [ ]:
!find /content/yolo_coffee_room -maxdepth 3 -type f | head -30
!cat /content/yolo_coffee_room/data.yaml

## 6. Dem so anh va label

In [ ]:
from pathlib import Path

root = Path('/content/yolo_coffee_room')

train_images = list((root / 'images' / 'train').glob('*.jpg'))
val_images = list((root / 'images' / 'val').glob('*.jpg'))
train_labels = list((root / 'labels' / 'train').glob('*.txt'))
val_labels = list((root / 'labels' / 'val').glob('*.txt'))

print('Train images:', len(train_images))
print('Train labels:', len(train_labels))
print('Val images:', len(val_images))
print('Val labels:', len(val_labels))

## 7. Train YOLOv8n

Co the tang/giam `epochs` tuy theo thoi gian train.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/yolo_coffee_room/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    name='fall_detection_yolov8n'
)

## 8. Kiem tra file model sau train

File can tai ve la `best.pt`.

In [ ]:
!ls -lh /content/runs/detect/fall_detection_yolov8n/weights

## 9. Danh gia nhanh model tren validation set

In [ ]:
from ultralytics import YOLO

best_model = YOLO('/content/runs/detect/fall_detection_yolov8n/weights/best.pt')
metrics = best_model.val(data='/content/yolo_coffee_room/data.yaml')

## 10. Tai best.pt ve may

Sau khi tai ve, dat file vao project local:

```text
models/best.pt
```

In [ ]:
from google.colab import files

files.download('/content/runs/detect/fall_detection_yolov8n/weights/best.pt')

# Chay demo video tren Colab

Phan nay dung khi ban da co:
- `best.pt`
- `test_fall.mp4`
- `predict_video.py`

Neu vua train xong trong notebook nay thi `best.pt` da co san o `/content/runs/detect/.../best.pt`.

## 11. Tao cau truc demo

In [ ]:
!mkdir -p /content/fall-demo/models
!mkdir -p /content/fall-demo/videos/input
!mkdir -p /content/fall-demo/videos/output
!mkdir -p /content/fall-demo/src

## 12. Copy model vua train sang thu muc demo

In [ ]:
import shutil
from pathlib import Path

root = Path('/content/fall-demo')
shutil.copy('/content/runs/detect/fall_detection_yolov8n/weights/best.pt', root / 'models' / 'best.pt')
print('Da copy best.pt')

## 13. Upload video test va file predict_video.py

Upload 2 file:
- `test_fall.mp4`
- `predict_video.py`

File `predict_video.py` lay tu project local:

```text
src/predict_video.py
```

In [ ]:
from google.colab import files

uploaded_demo = files.upload()
print(uploaded_demo.keys())

## 14. Dua file demo vao dung vi tri

In [ ]:
import shutil
from pathlib import Path

root = Path('/content/fall-demo')

shutil.copy('test_fall.mp4', root / 'videos' / 'input' / 'test_fall.mp4')
shutil.copy('predict_video.py', root / 'src' / 'predict_video.py')

print('Da chuan bi file demo')

## 15. Chay demo phat hien te nga

In [ ]:
%cd /content/fall-demo
!python src/predict_video.py

## 16. Tai video ket qua ve may

Video output:

```text
/content/fall-demo/videos/output/result_fall.mp4
```

In [ ]:
from google.colab import files

files.download('/content/fall-demo/videos/output/result_fall.mp4')